### Structured Output

Models can be requested to provide their response in a formate matching a given schema. This is usefull for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema tupes and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, description and nested structure.

In [23]:
from langchain.chat_models import init_chat_model
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002CFC00027D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002CFC059E950>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [24]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")   
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")


In [25]:
## Using this function we allow model to follow this validation class of pydantic which is Movie in this case

model_with_structure= model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002CFC00027D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002CFC059E950>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of

In [26]:
model.invoke("provide the details about Iception movie?")

AIMessage(content='<think>\nOkay, the user asked for details about the movie "Iception". First, I need to confirm the correct title. "Iception" doesn\'t ring a bell. The correct title is probably "Inception" (inception is the act of starting something). Maybe the user made a typo.\n\nSo, assuming they meant "Inception", I should provide the correct information. Let me recall the movie. Directed by Christopher Nolan, released in 2010. Starring Leonardo DiCaprio as Dom Cobb, the main character. The plot involves entering dreams to plant ideas, called "inception". There\'s a lot of action, visual effects, and a complex narrative. The cast includes Ellen Page, Joseph Gordon-Levitt, Tom Hardy, and others.\n\nI should mention the director, release year, main actors, plot summary, themes like reality vs. dreams, the concept of planting ideas, the use of time dilation in dreams, and the ending which is ambiguous. Also, the critical reception and awards it received, like four Oscars. Maybe touc

In [27]:
model_with_structure.invoke("provide the details about Iception movie?")


Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message raw output alongside parsed structure

In [28]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")   
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("provide details about the movie Inceptioin")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception". Let me see what I need to do here. The available tool is the Movie function, which requires the title, year, director, and rating. First, I need to recall the information about Inception. The title is "Inception", directed by Christopher Nolan. It was released in 2010. The rating... maybe around 8.8 on IMDb? Wait, I should confirm that. But since I can\'t look it up, I\'ll go with what I remember. The user wants the details, so I need to structure the response using the Movie function. Let me make sure all required parameters are included: title, year, director, rating. Yep, that\'s all there. I\'ll format the JSON accordingly.\n', 'tool_calls': [{'id': '94prd9z2q', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'comp

### Nesteds structure

In [29]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int 
    cast:list[Actor]  ## this is nested structure where we have list of actors and each actor has name and role
    genres:list[str]
    budget:float | None = Field(default=None, description="The budget of the movie in millions")

    

In [30]:
model_with_structure= model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("provide details about the movie Inception?")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Bard')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)


### TypedDict

TypeDict provides a simpler alrenative using python's built-in typing, ideasl when you don't need runtime validation.

Example., runtime validation mean as in pydantic we can also validate like this (title:str)  title should be in string if not it's not validate..

In [31]:
from typing_extensions import Annotated, TypedDict

## TypedDict is used to define the structure of the output in a more flexible way than pydantic models. It allows us to define a dictionary with specific keys and value types without the need for creating a class.
## Annotated is used to add metadata to the types defined in TypedDict. This metadata can be used by the model to understand the structure of the output better and generate more accurate responses.

In [32]:
class MovieDict(TypedDict):
    """Movie details in a dictionary format"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movies rating out of 10"]

In [33]:
model_withtypeddict = model.with_structured_output(MovieDict)
response = model_withtypeddict.invoke("please provide the details of the movie Avengers?")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [44]:
from pprint import pprint
class Actor(TypedDict):
    name:str
    role:str
class MovieDetails(TypedDict):
    title:str
    year:int 
    cast:list[Actor]  
    genres:list[str]
    budget:float | None = Field(default=None, description="The budget of the movie in millions")

    model_with_structure= model.with_structured_output(MovieDetails)

    response = model_with_structure.invoke("please provide the details of the movie Avengers?")
    pprint(response)

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
          {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
          {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
          {'name': 'Chris Hemsworth', 'role': 'Thor'},
          {'name': 'Scarlett Johansson',
           'role': 'Natasha Romanoff / Black Widow'},
          {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}


In [45]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren't any restriction. you create it using the @dataclass decorator.

#### ==>  Let's compare all three class using agent 

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class Contactinfo(BaseModel):
    """contact information of a person"""
    name:str=Field(description="The name of the contact")
    email:str=Field(description="The email of the contact")
    phone:str=Field(description="The phone number of the contact"
    )
                    
agent = create_agent(
    model,
    response_format=Contactinfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Please provide the contact information for John Doe, john@example.com, (555) 123 4567."}]
})
result

{'messages': [HumanMessage(content='Please provide the contact information for John Doe, john@example.com, (555) 123 4567.', additional_kwargs={}, response_metadata={}, id='58d963c4-df48-4eda-b873-a794fcdd5f96'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the contact information of John Doe. Let me check the tools provided. There's a function called Contactinfo that requires name, email, and phone. The user provided all three: name is John Doe, email is john@example.com, and phone is (555) 123 4567. I need to make sure all required parameters are included. They are. So I should call the Contactinfo function with these details. No missing info, so the tool call is straightforward.\n", 'tool_calls': [{'id': 'px8y30nra', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123 4567"}', 'name': 'Contactinfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 160, 'prompt_toke

In [53]:
result['structured_response']

Contactinfo(name='John Doe', email='john@example.com', phone='(555) 123 4567')

In [54]:
from langchain.agents import create_agent

class Contactinfo(TypedDict):
    """contact information of a person"""
    name:str
    email:str
    phone:str
                    
agent = create_agent(
    model,
    response_format=Contactinfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Please provide the contact information for John Doe, john@example.com, (555) 123 4567."}]
})
result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123 4567'}

In [55]:
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class Contactinfo:
    """contact information of a person"""
    name:str
    email:str
    phone:str
                    
agent = create_agent(
    model,
    response_format=Contactinfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Please provide the contact information for John Doe, john@example.com, (555) 123 4567."}]
})
result['structured_response']

Contactinfo(name='John Doe', email='john@example.com', phone='(555) 123 4567')